In [1]:
# =========================================================================
# CREDIT RISK DEMO
# LOAD PKL + ENTER ONE CUSTOMER + PREDICT CREDIT RISK
# =========================================================================

import os
import joblib
import numpy as np
import pandas as pd


# =========================================================================
# 1. PATHS
# =========================================================================

BASE_DIR = r"D:\Thesis\8-PKL-file"

PKL_PATH = os.path.join(
    BASE_DIR,
    "credit_risk_package.pkl"
)


# =========================================================================
# 2. LOAD COMPLETE PKL PACKAGE
# =========================================================================

if not os.path.exists(PKL_PATH):

    raise FileNotFoundError(
        f"PKL file not found:\n{PKL_PATH}"
    )


credit_risk_package = joblib.load(
    PKL_PATH
)


# =========================================================================
# 3. LOAD MODEL AND PREPROCESSING INFORMATION
# =========================================================================

final_model = (
    credit_risk_package["model"]
)

scalers = (
    credit_risk_package["scalers"]
)

log_features = (
    credit_risk_package["log_features"]
)

scale_features = (
    credit_risk_package["scale_features"]
)

binary_features = (
    credit_risk_package["binary_features"]
)

categorical_features = (
    credit_risk_package["categorical_features"]
)

final_features = (
    credit_risk_package["final_features"]
)

training_encoded_columns = (
    credit_risk_package[
        "training_encoded_columns"
    ]
)

best_threshold = (
    credit_risk_package[
        "best_threshold"
    ]
)

t1 = (
    credit_risk_package["t1"]
)

t2 = (
    credit_risk_package["t2"]
)

occupation_values = (
    credit_risk_package[
        "occupation_values"
    ]
)

product_type_values = (
    credit_risk_package[
        "product_type_values"
    ]
)

loan_intent_values = (
    credit_risk_package[
        "loan_intent_values"
    ]
)


# =========================================================================
# 4. ENTER CUSTOMER
# =========================================================================

customer = {

    "customer_id":
        "DEMO001",

    "age":
        40,

    "occupation_status":
        "Employed",

    "years_employed":
        10,

    "annual_income":
        60000,

    "credit_score":
        720,

    "credit_history_years":
        15,

    "savings_assets":
        10000,

    "current_debt":
        10000,

    "defaults_on_file":
        0,

    "delinquencies_last_2yrs":
        0,

    "derogatory_marks":
        0,

    "product_type":
        "Personal Loan",

    "loan_intent":
        "Home Improvement",

    "loan_amount":
        15000,

    "interest_rate":
        10,

    "debt_to_income_ratio":
        16.67,

    "loan_to_income_ratio":
        25,

    "payment_to_income_ratio":
        8

}


# =========================================================================
# 5. VALIDATE REQUIRED FIELDS
# =========================================================================

required_fields = [

    "customer_id",
    "age",
    "occupation_status",
    "years_employed",
    "annual_income",
    "credit_score",
    "credit_history_years",
    "savings_assets",
    "current_debt",
    "defaults_on_file",
    "delinquencies_last_2yrs",
    "derogatory_marks",
    "product_type",
    "loan_intent",
    "loan_amount",
    "interest_rate",
    "debt_to_income_ratio",
    "loan_to_income_ratio",
    "payment_to_income_ratio"

]


missing_fields = [

    field

    for field
    in required_fields

    if field not in customer

]


if missing_fields:

    raise ValueError(
        f"Missing customer fields: "
        f"{missing_fields}"
    )


# =========================================================================
# 6. CREATE RAW CUSTOMER DATAFRAME
# =========================================================================

customer_id = (
    customer["customer_id"]
)


raw_customer = pd.DataFrame(
    [customer]
)


# =========================================================================
# 7. DATA QUALITY VALIDATION
# =========================================================================

validation_errors = []


# ---------------------------------------------------------
# Missing values
# ---------------------------------------------------------

if raw_customer[
    required_fields
].isna().any().any():

    missing_value_columns = (
        raw_customer[
            required_fields
        ]
        .columns[
            raw_customer[
                required_fields
            ]
            .isna()
            .any()
        ]
        .tolist()
    )

    validation_errors.append(
        f"Missing values in: "
        f"{missing_value_columns}"
    )


# ---------------------------------------------------------
# Age
# ---------------------------------------------------------

if (
    customer["age"] < 18
    or
    customer["age"] > 100
):

    validation_errors.append(
        "Age must be between 18 and 100."
    )


# ---------------------------------------------------------
# Credit score
# ---------------------------------------------------------

if (
    customer["credit_score"] < 300
    or
    customer["credit_score"] > 850
):

    validation_errors.append(
        "Credit score must be between 300 and 850."
    )


# ---------------------------------------------------------
# Years employed
# ---------------------------------------------------------

if customer[
    "years_employed"
] < 0:

    validation_errors.append(
        "Years employed cannot be negative."
    )


if customer[
    "years_employed"
] > (
    customer["age"] - 18
):

    validation_errors.append(
        "Years employed exceeds the "
        "possible employment period."
    )


# ---------------------------------------------------------
# Credit history
# ---------------------------------------------------------

if customer[
    "credit_history_years"
] < 0:

    validation_errors.append(
        "Credit history cannot be negative."
    )


if customer[
    "credit_history_years"
] > (
    customer["age"] - 18
):

    validation_errors.append(
        "Credit history exceeds the "
        "possible period."
    )


# ---------------------------------------------------------
# Annual income
# ---------------------------------------------------------

if customer[
    "annual_income"
] <= 0:

    validation_errors.append(
        "Annual income must be greater than zero."
    )


# ---------------------------------------------------------
# Financial variables
# ---------------------------------------------------------

if customer[
    "savings_assets"
] < 0:

    validation_errors.append(
        "Savings/assets cannot be negative."
    )


if customer[
    "current_debt"
] < 0:

    validation_errors.append(
        "Current debt cannot be negative."
    )


if customer[
    "loan_amount"
] < 0:

    validation_errors.append(
        "Loan amount cannot be negative."
    )


# ---------------------------------------------------------
# Interest rate
# ---------------------------------------------------------

if (
    customer["interest_rate"] < 0
    or
    customer["interest_rate"] > 100
):

    validation_errors.append(
        "Interest rate must be between 0 and 100."
    )


# ---------------------------------------------------------
# Counts
# ---------------------------------------------------------

if customer[
    "defaults_on_file"
] < 0:

    validation_errors.append(
        "Defaults on file cannot be negative."
    )


if customer[
    "delinquencies_last_2yrs"
] < 0:

    validation_errors.append(
        "Delinquencies cannot be negative."
    )


if customer[
    "derogatory_marks"
] < 0:

    validation_errors.append(
        "Derogatory marks cannot be negative."
    )


# ---------------------------------------------------------
# Ratios
# ---------------------------------------------------------

if customer[
    "debt_to_income_ratio"
] < 0:

    validation_errors.append(
        "Debt-to-income ratio cannot be negative."
    )


if customer[
    "loan_to_income_ratio"
] < 0:

    validation_errors.append(
        "Loan-to-income ratio cannot be negative."
    )


if customer[
    "payment_to_income_ratio"
] < 0:

    validation_errors.append(
        "Payment-to-income ratio cannot be negative."
    )


# =========================================================================
# 8. CHECK CATEGORICAL VALUES
# =========================================================================

if (
    customer["occupation_status"]
    not in occupation_values
):

    validation_errors.append(
        f"Unknown occupation_status: "
        f"{customer['occupation_status']}. "
        f"Expected one of: "
        f"{occupation_values}"
    )


if (
    customer["product_type"]
    not in product_type_values
):

    validation_errors.append(
        f"Unknown product_type: "
        f"{customer['product_type']}. "
        f"Expected one of: "
        f"{product_type_values}"
    )


if (
    customer["loan_intent"]
    not in loan_intent_values
):

    validation_errors.append(
        f"Unknown loan_intent: "
        f"{customer['loan_intent']}. "
        f"Expected one of: "
        f"{loan_intent_values}"
    )


# =========================================================================
# 9. STOP IF INPUT IS INVALID
# =========================================================================

if validation_errors:

    print("\n")
    print("=" * 75)
    print("DATA VALIDATION FAILED")
    print("=" * 75)

    for error in validation_errors:

        print(
            f"❌ {error}"
        )

    print("=" * 75)

    raise SystemExit


# =========================================================================
# 10. DUPLICATE CHECK
# =========================================================================

duplicate_check = (
    raw_customer
    .duplicated()
    .iloc[0]
)


# =========================================================================
# 11. IMPORTANT:
# RE-CALCULATE RATIOS FROM RAW FINANCIAL INFORMATION
# =========================================================================

# This ensures consistency between income, debt and loan amount.

calculated_dti = (
    customer["current_debt"]
    /
    customer["annual_income"]
)


calculated_lti = (
    customer["loan_amount"]
    /
    customer["annual_income"]
)


# Use the calculated values for inference.
raw_customer[
    "debt_to_income_ratio"
] = calculated_dti


raw_customer[
    "loan_to_income_ratio"
] = calculated_lti


# =========================================================================
# 12. REMOVE CUSTOMER ID
# =========================================================================

X_new = raw_customer.drop(
    columns=["customer_id"]
).copy()


# =========================================================================
# 13. LOG TRANSFORMATION
# =========================================================================

for col in log_features:

    X_new[
        f"{col}_log"
    ] = np.log1p(
        X_new[col]
    )


# =========================================================================
# 14. SCALE ORIGINAL NUMERICAL FEATURES
# =========================================================================

for col in scale_features:

    scaler = scalers[col]

    X_new[
        f"{col}_scaled"
    ] = scaler.transform(
        X_new[[col]]
    )


# =========================================================================
# 15. SCALE LOG FEATURES
# =========================================================================

log_scaled_features = [

    f"{col}_log"

    for col
    in log_features

]


for col in log_scaled_features:

    scaler = scalers[col]

    X_new[
        f"{col}_scaled"
    ] = scaler.transform(
        X_new[[col]]
    )


# =========================================================================
# 16. BINARY FEATURES
# =========================================================================

for col in binary_features:

    X_new[
        f"{col}_binary"
    ] = (
        X_new[col] > 0
    ).astype(int)


# =========================================================================
# 17. ONE-HOT ENCODING
# =========================================================================

X_new_encoded = pd.get_dummies(

    X_new,

    columns=
        categorical_features,

    drop_first=True

)


# =========================================================================
# 18. MAKE EXACT SAME COLUMNS AS TRAINING
# =========================================================================

X_new_encoded = (
    X_new_encoded
    .reindex(
        columns=
            training_encoded_columns,
        fill_value=0
    )
)


# =========================================================================
# 19. DROP ORIGINAL FEATURES
# =========================================================================

features_to_drop = (
    log_features
    +
    scale_features
    +
    binary_features
)


X_new_encoded = (
    X_new_encoded
    .drop(
        columns=
            features_to_drop,
        errors="ignore"
    )
)


# =========================================================================
# 20. SELECT EXACT FINAL FEATURES
# =========================================================================

X_new_final = (
    X_new_encoded[
        final_features
    ]
)


# =========================================================================
# 21. FINAL SAFETY CHECK
# =========================================================================

missing_model_features = [

    feature

    for feature
    in final_features

    if feature
    not in X_new_final.columns

]


if missing_model_features:

    raise ValueError(
        "Missing model features: "
        f"{missing_model_features}"
    )


# =========================================================================
# 22. MODEL PREDICTION
# =========================================================================

risk_probability = float(

    final_model
    .predict_proba(
        X_new_final
    )[0, 1]

)


# =========================================================================
# 23. THREE-ZONE DECISION
# =========================================================================

if risk_probability < t1:

    risk_level = (
        "LOW RISK"
    )

    decision = (
        "SAFE - AUTO ACCEPT"
    )

    decision_explanation = (
        "The predicted risk probability "
        "is below the lower threshold."
    )


elif risk_probability < t2:

    risk_level = (
        "MEDIUM / UNCERTAIN RISK"
    )

    decision = (
        "HUMAN REVIEW"
    )

    decision_explanation = (
        "The predicted risk probability "
        "falls inside the uncertainty zone."
    )


else:

    risk_level = (
        "HIGH RISK"
    )

    decision = (
        "RISKY - AUTO REJECT"
    )

    decision_explanation = (
        "The predicted risk probability "
        "is above the upper threshold."
    )


# =========================================================================
# 24. DISPLAY RESULT
# =========================================================================

print("\n")
print("=" * 75)
print("             CREDIT RISK DECISION SUPPORT SYSTEM")
print("=" * 75)

print(
    f"Customer ID:             {customer_id}"
)

print(
    f"Duplicate input:         {duplicate_check}"
)

print(
    "Missing values:          0"
)

print(
    f"Calculated DTI:          "
    f"{calculated_dti:.4f}"
)

print(
    f"Calculated LTI:          "
    f"{calculated_lti:.4f}"
)

print("-" * 75)

print(
    "MODEL PREDICTION"
)

print("-" * 75)

print(
    f"Risk probability:        "
    f"{risk_probability:.2%}"
)

print(
    f"Lower threshold (t1):    "
    f"{t1:.2%}"
)

print(
    f"Upper threshold (t2):    "
    f"{t2:.2%}"
)

print(
    f"Risk level:              "
    f"{risk_level}"
)

print()

print(
    f"FINAL DECISION:          "
    f"{decision}"
)

print()

print(
    decision_explanation
)

print("=" * 75)


# =========================================================================
# 25. EXPLICIT DECISION MESSAGE
# =========================================================================

if risk_probability < t1:

    print()
    print(
        "🟢 SAFE CUSTOMER"
    )

    print(
        "Recommendation: "
        "AUTO-ACCEPT"
    )


elif risk_probability < t2:

    print()
    print(
        "🟡 HUMAN REVIEW REQUIRED"
    )

    print(
        "Recommendation: "
        "SEND TO HUMAN CREDIT ANALYST"
    )


else:

    print()
    print(
        "🔴 RISKY CUSTOMER"
    )

    print(
        "Recommendation: "
        "AUTO-REJECT"
    )


print()
print(
    "The prediction is generated by the "
    "trained XGBoost model saved in the PKL package."
)

print(
    "This system is a decision-support demonstration "
    "and not a real-world lending decision."
)



             CREDIT RISK DECISION SUPPORT SYSTEM
Customer ID:             DEMO001
Duplicate input:         False
Missing values:          0
Calculated DTI:          0.1667
Calculated LTI:          0.2500
---------------------------------------------------------------------------
MODEL PREDICTION
---------------------------------------------------------------------------
Risk probability:        99.83%
Lower threshold (t1):    20.00%
Upper threshold (t2):    60.00%
Risk level:              HIGH RISK

FINAL DECISION:          RISKY - AUTO REJECT

The predicted risk probability is above the upper threshold.

🔴 RISKY CUSTOMER
Recommendation: AUTO-REJECT

The prediction is generated by the trained XGBoost model saved in the PKL package.
This system is a decision-support demonstration and not a real-world lending decision.
